In [ ]:
import os
import getpass
import boto3
from botocore.exceptions import ClientError
import time

def check_instance_status(ec2, instance_id):
    """Fetch live status of a specific instance safely."""
    if not instance_id:
        return None
    try:
        response = ec2.describe_instances(InstanceIds=[instance_id])
        instance = response['Reservations'][0]['Instances'][0]
        name = "Unnamed"
        for tag in instance.get('Tags', []):
            if tag['Key'] == 'Name':
                name = tag['Value']
                break
        return {
            'id': instance['InstanceId'],
            'name': name,
            'state': instance['State']['Name'],
            'public_ip': instance.get('PublicIpAddress', 'No Public IP')
        }
    except ClientError:
        return None

def run_ssm_command(ssm, instance_id, command):
    """Executes a shell command inside the EC2 instance using AWS SSM."""
    try:
        response = ssm.send_command(
            InstanceIds=[instance_id],
            DocumentName="AWS-RunShellScript",
            Parameters={'commands': [command]}
        )
        command_id = response['Command']['CommandId']
        time.sleep(2)
        
        output = ssm.get_command_invocation(
            CommandId=command_id,
            InstanceId=instance_id
        )
        return output.get('StandardOutputContent', '').strip(), output.get('StandardErrorContent', '').strip()
    except ClientError as e:
        return "", f"SSM Error: {e.response['Error']['Message']}"

def main():
    print("=== AWS EC2 Automation Suite ===")
    key_id = input("AWS Access Key ID: ").strip()
    secret_key = getpass.getpass("AWS Secret Key: ").strip()
    region = input("Region (default: ap-south-1): ").strip().lower() or "ap-south-1"

    try:
        endpoint = f"https://ec2.{region}.amazonaws.com"
        ec2 = boto3.client('ec2', aws_access_key_id=key_id, aws_secret_access_key=secret_key, region_name=region, endpoint_url=endpoint)
        ssm = boto3.client('ssm', aws_access_key_id=key_id, aws_secret_access_key=secret_key, region_name=region)
        print("Authenticated successfully.")
    except Exception as e:
        print(f"Initialization Failed: {str(e)}")
        return

    target_instance_id = input("\nEnter Target Instance ID (or press Enter to skip): ").strip()

    while True:
        status = check_instance_status(ec2, target_instance_id)
        
        print("\n" + "="*40)
        if status:
            print(f"Target Instance : {target_instance_id} ({status['name']})")
            print(f"Current State   : {status['state'].upper()}")
            print(f"Public IP       : {status['public_ip']}")
        else:
            print(f"Target Instance : {target_instance_id or 'None'}")
            print("Tip: Use Option 2 to launch an instance.")
        print("="*40)
        
        print("1. Change Target Instance ID")
        print("2. Launch New Instance")
        print("3. Start Target Instance")
        print("4. Stop Target Instance")
        print("5. View Directory Files")
        print("6. Create/Update File")
        print("7. Delete File")
        print("8. Exit")
        print("="*40)
        
        choice = input("\nSelect an option (1-8): ").strip()

        if choice == '1':
            target_instance_id = input("Enter Target Instance ID: ").strip()
            print("Target updated.")

        elif choice == '2':
            print("\n--- Launch Instance ---")
            ami_id = input("Enter AMI ID: ").strip()
            inst_type = input("Instance Type (default: t2.micro): ").strip() or "t2.micro"
            inst_name = input("Name Tag (default: Automated-Server): ").strip() or "Automated-Server"
            key_name = input("Key Pair Name (optional): ").strip()
            
            if not ami_id:
                print("Error: AMI ID is required.")
                continue
            try:
                launch_params = {
                    'ImageId': ami_id,
                    'InstanceType': inst_type,
                    'MinCount': 1,
                    'MaxCount': 1,
                    'TagSpecifications': [{'ResourceType': 'instance', 'Tags': [{'Key': 'Name', 'Value': inst_name}]}]
                }
                if key_name:
                    launch_params['KeyName'] = key_name

                new_inst = ec2.run_instances(**launch_params)
                target_instance_id = new_inst['Instances'][0]['InstanceId']
                print(f"Launched. Current target: {target_instance_id}")
                print("Note: Initialization takes a moment.")
            except ClientError as e:
                print(f"Launch Failed: {e.response['Error']['Message']}")

        elif choice == '3':
            if not target_instance_id:
                print("Error: No target instance specified.")
                continue
            try:
                ec2.start_instances(InstanceIds=[target_instance_id])
                print("Start command sent.")
            except ClientError as e:
                print(f"Failed: {e.response['Error']['Message']}")

        elif choice == '4':
            if not target_instance_id:
                print("Error: No target instance specified.")
                continue
            if input(f"Stop instance {target_instance_id}? (y/N): ").lower() == 'y':
                try:
                    ec2.stop_instances(InstanceIds=[target_instance_id])
                    print("Stop command sent.")
                except ClientError as e:
                    print(f"Failed: {e.response['Error']['Message']}")

        elif choice in ['5', '6', '7']:
            if not target_instance_id:
                print("Error: Specify or launch an instance first.")
                continue
            if not status or status['state'] != 'running':
                print("Error: Instance must be running.")
                continue

            if choice == '5':
                path = input("Directory path (default: /home/ubuntu): ").strip() or "/home/ubuntu"
                out, err = run_ssm_command(ssm, target_instance_id, f"ls -la {path}")
                if err: print(f"Failed:\n{err}")
                else: print(f"\nFiles in {path}:\n{out}")

            elif choice == '6':
                file_path = input("File path (e.g. /home/ubuntu/note.txt): ").strip()
                if not file_path: continue
                content = input("Content: ")
                safe_content = content.replace("'", "'\\''")
                out, err = run_ssm_command(ssm, target_instance_id, f"echo '{safe_content}' > {file_path}")
                if err: print(f"Failed:\n{err}")
                else: print(f"File updated: {file_path}")

            elif choice == '7':
                file_path = input("File path to delete: ").strip()
                if not file_path: continue
                if input(f"Delete {file_path} permanently? (y/N): ").lower() == 'y':
                    out, err = run_ssm_command(ssm, target_instance_id, f"rm -f {file_path}")
                    if err: print(f"Failed:\n{err}")
                    else: print("File deleted.")

        elif choice == '8':
            print("Exiting.")
            break
        else:
            print("Invalid option.")

if __name__ == "__main__":
    main()